## 第 6 课：Block MMA

> 对应原文：笔记 (5)。把问题规模从 Tile 扩展到 Block（128×128×64），并解析 TiledCopy、TiledMMA 在 Block 维度下的分块特点。

## 学习目标

- 理解为什么需要以 Tile 为单位循环执行 Copy 和 MMA；
- 弄清 MMA 与 Copy 分块粒度的不同：**Copy 按 Tile 粒度循环、MMA 按 Atom 粒度循环**；
- 会读 Block 维度下 partition 后 Tensor 的 metadata；
- 理解编译器对 copy/mma 指令的重排，以及手动控制循环的意义。

## 1. Tile 扩展至 Block

### 1.1 为什么不能一次装下整个 Block？

如果完整加载 Block（128×128×64）到寄存器：

In [ ]:
128 × 128 × 64 / 2 = 524288 个寄存器 >> 32768 上限

必然寄存器溢出 → 使用大量 GMEM、性能大幅下降。所以要把 Block 沿 (M, N, K) 切成若干 Tile，用**三重循环**依次处理每个 Tile 的 Copy 和 MMA：

In [ ]:
for (int m_tile = 0; m_tile < NTilesM; ++m_tile) {
    for (int n_tile = 0; n_tile < NTilesN; ++n_tile) {
        for (int k_tile = 0; k_tile < NTilesK; ++k_tile) {
            copy(tiled_copy, gA(_, m_tile, k_tile), rA(_, m_tile, k_tile));
            copy(tiled_copy, gB(_, n_tile, k_tile), rB(_, n_tile, k_tile));
            gemm(tiled_mma, rC, rA, rB, rC);
        }
    }
}

理论上一个 block 能处理的规模没有限制。但循环中部分 Tile 被重复读取（Tile1 和 Tile2 共用同一份 A 数据），增加 GMEM 访存——下一篇用 SMEM 解决。

![Tile 访存存在重复读取 GMEM 的问题](assets/figs/fig_01_Tile_访存存在重复读取_GMEM_的问题.png)

![Block MMA 示意图](assets/figs/fig_01_Block_MMA_示意图.png)

## 2. Block MMA 实现

本篇规格：MMA 指令升级为 16×8×16，Tile 32×32×32，Block 128×128×64（(M,N,K) 各扩展 (4,4,2) 倍），256 线程。

Spec 层只需把 kTile 改为 Block 大小：

In [ ]:
template <typename OutType_, typename ComputeTypeA_, typename ComputeTypeB_, typename ComputeTypeC_,
          int kTileM_ = 128, int kTileN_ = 128, int kTileK_ = 64>
struct KernelSpec { ... };

### 2.1 MMA、Copy 分块 Tensor 的扩展维度

**TiledMMA 分块**（扩展到 Block 后）：

In [ ]:
# 上篇（Tile 32×32×16）
gmem_ptr[16b](0x...) o ((_2,_2),_1,_2):((_1,128),_0,_8)
...
# 本篇（Block 128×128×64）
gmem_ptr[16b](0x...) o ((_2,_2,_2),_4,_4):((_1,512,_8),2048,_16)
ptr[16b](0x...) o ((_2,_2,_2),_4,_4):((_1,_2,_4),_32,_8)

变化：MMA 指令从 16×8×8 变 16×8×16，与 K 相关的 MMA（单指令单线程数据量）扩大 2 倍；`(MMA_M, MMA_N, MMA_K)` 从 `(1,1,2)` 变 `(4,4,4)`。

**关键结论**：MMA 分块以 **MMA Atom 为粒度**，`(MMA_M, MMA_N, MMA_K)` 既包含 Atom→Tile 的扩展，也包含 Tile→Block 的扩展。

**TiledCopy 分块**（retile 后）：

In [ ]:
# 上篇
gmem_ptr[16b](0x...) o ((_1,(_2,_2,_2)),_1,_1):((_0,(_1,128,_8)),_0,_0)
# 本篇
gmem_ptr[16b](0x...) o ((_1,(_2,_2,_4)),_4,_2):((_0,(_1,512,_8)),2048,_32)

CPY（单 Tile 单线程拷贝数据量）因 Tile 扩大一倍；`(CPY_M, CPY_N, CPY_K) = (4,4,2)`，等于 Tile→Block 的扩展维度。

**关键结论**：Copy 分块以 **Tile 为粒度**，CPY_* 只表示 Tile→Block 的扩展。

两个"粒度不同"决定了循环方式不同（见下节），也是 `retile_S/D` 需要重排 Layout 的原因。

### 2.2 循环执行 Copy 和 MMA

- Copy：按 Tile 索引 `(m_tile, n_tile, k_tile)`；
- MMA：按 Atom 扩展维度 `(im, in, ik)`（范围是 kMmaValExpand 的整数倍区间）。

In [ ]:
for (int m_tile = 0; m_tile < NTilesM; ++m_tile) {
    for (int n_tile = 0; n_tile < NTilesN; ++n_tile) {
        for (int k_tile = 0; k_tile < NTilesK; ++k_tile) {
            copy(g2r_tiled_copy_a, tAgA(_, m_tile, k_tile), tArA(_, m_tile, k_tile));
            copy(g2r_tiled_copy_b, tBgB(_, n_tile, k_tile), tBrB(_, n_tile, k_tile));

            for (int im = m_tile*kMmaValExpandM; im < (m_tile+1)*kMmaValExpandM; ++im)
            for (int in = n_tile*kMmaValExpandN; in < (n_tile+1)*kMmaValExpandN; ++in)
            for (int ik = k_tile*kMmaValExpandK; ik < (k_tile+1)*kMmaValExpandK; ++ik) {
                gemm(tiled_mma, tCrC(_, im, in), tCrA(_, im, ik), tCrB(_, in, ik), tCrC(_, im, in));
            }
        }
    }
}

也可以完全交给 API（内部用蛇形遍历算法最大化 Cache 复用）：

In [ ]:
copy(g2r_tiled_copy_a, tAgA, tArA);
copy(g2r_tiled_copy_b, tBgB, tBrB);
gemm(tiled_mma, tCrC, tCrA, tCrB, tCrC);

或手动控制部分维度（例如只控制 K）：

In [ ]:
for (int ik = 0; ik < NTilesK; ++ik) {
    copy(g2r_tiled_copy_a, tAgA(_, _, ik), tArA(_, _, ik));
    copy(g2r_tiled_copy_b, tBgB(_, _, ik), tBrB(_, _, ik));
    for (int gk = ik*kMmaValExpandK; gk < (ik+1)*kMmaValExpandK; ++gk) {
        gemm(tiled_mma, tCrC, tCrA(_, _, gk), tCrB(_, _, gk), tCrC);
    }
}

### 2.3 PTX / SASS：编译器重排

疑问：最简写法是否意味着等整个 Block 数据拷贝完才算 mma？会寄存器溢出吗？

**不会**：编译器会合理重排 copy 与 mma 指令，在充分利用寄存器的前提下交错执行。

那手动控制循环的意义？编译器不是全能的：

- 手动循环可以控制 copy/mma 的流水，让拷贝尽量被 mma 计算掩盖（微小性能提升）；
- 对编译器无法自动优化的 **SMEM 流水线**，必须手动写循环（下一篇的主题）。

![Block MMA 的 PTX / SASS code](assets/figs/fig_03_Block_MMA_的_PTX___SASS_code.png)

## 同时回答

1. 为什么不能把整个 Block 一次性加载到寄存器？写出 128×128×64 的寄存器需求计算。
2. Copy 按什么粒度循环、MMA 按什么粒度循环？两者的扩展维度含义有什么不同？
3. 最简写法下编译器会做什么？那手动控制循环的意义在哪里（特别是为下一篇 SMEM 流水线埋伏笔的点）？

把代码和三个答案发给我，我继续审查。